### Ingesting Drivers File

1) Read the file using spark dataframe reader API
2) Defined and enforced schema (preserved the nested structure)
3) Added metadata columns or audit columns such as:

    -- Source File

    --Ingestion Timestamp

4) Wrote the tables into to Bronze Layer    

Reading the CSV file

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
%run "../00-Common/02. Bronze-helper_func"

In [0]:
# Defining relevant Source and Table name

Source_file=f"{landing_folder_path}/drivers.json"
Table_name=f"{catalog_name}.{bronze_schema}.drivers"

This dataset is in nested json format. So first, nested json will be addressed and the schema will be dinfed for that followed by other objects.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType


name_Schema=StructType([
                        StructField('givenName',StringType()),
                        StructField('familyName', StringType()) 
  ])

drivers_Schema=StructType([
                          StructField("driverId",StringType()),
                          StructField("name",name_Schema),
                          StructField("dateofBirth",DateType()),
                          StructField("nationality",StringType()),
                          StructField("url",StringType())
])


In [0]:
drivers_df=(
                spark.read.format('json')
                #.option('header',True)
                #.option('inferSchema','true')
                .option('mode','FAILFAST')      # PERMISSIVE, DROPMALFORMED, FAILFAST
                .schema(drivers_Schema)
                .load(Source_file))

In [0]:
display(drivers_df)

2) Adding Metadata

In [0]:
Drivers_final_df= add_ingestion_metadata(drivers_df)

In [0]:
display(Drivers_final_df)

3) Witting data to Bronze layer/delta table

In [0]:
(
    Drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(Table_name)
)

In [0]:
display(spark.table(Table_name))